# Mission 02 - FST Dialogue Systems, Reasoning Evaluation & Noisy Channels

**Squad:** Sutskever (Group 18) | **Role:** Core Architect / Systems Architect / Lead Engineer | **Author:** Syed Hashir Sohail

This notebook drives `04_fst_noisy_simulator.py`, which holds all the logic:

| Part | What it shows |
|---|---|
| 1 | The FST dialogue manager walking its states and filling slots |
| 2 | The noisy channel: word counter, noise-token injector (append / substitute), entropy |
| 3 | The five stress tests (T1-T5) against the FST |
| 4 | The live 8-turn Anti-Turing interrogation with Gemini through the channel |
| 5 | Saving `05_Noisy_Interrogation_Transcript.md` (existing files are never overwritten; a new run becomes `_run2`, `_run3`, ...) |

**Before running:** put `GEMINI_API_KEY` (and optionally `GEMINI_MODEL`) in a local `.env` file. Never commit it.
Use *Restart Kernel and Run All* so every output is fresh.

## 0. Setup

In [ ]:
import importlib.util, os, sys
from pathlib import Path

HERE = Path.cwd()  # run the notebook from the folder that contains 04_fst_noisy_simulator.py
spec = importlib.util.spec_from_file_location("sim", HERE / "04_fst_noisy_simulator.py")
sim = importlib.util.module_from_spec(spec)
sys.modules["sim"] = sim
spec.loader.exec_module(sim)
sim.load_dotenv()

FST = sim.load_fst_class()      # baseline class, or the squad's final class via FST_MODULE
SEED = 42
NOISE_MODE = "append"           # "append" (6 words + noise token) or "substitute" (7 words, one replaced)
HISTORY = "raw"                 # what the model sees of its own earlier replies: "raw" or "noisy"
USE_PERSONA = True             # False = send the prompts with no system prompt (full-length answers, baseline)
SUFFIX = ""                    # e.g. "_baseline" so a second run does not overwrite the first transcript
PROVIDER = os.environ.get("NOTEBOOK_PROVIDER", "auto")

print("FST class:", FST.__module__ + "." + FST.__name__)
print("Noise vocabulary:", sim.NOISE_VOCABULARY)

## 1. FST dialogue manager

Happy path: `INIT -> GREETING -> COLLECT_SLOTS -> CONFIRMATION -> TERMINAL`, with the slots `destination`, `speed` and `clearance` filled in order.

In [ ]:
fst = FST()
crew_inputs = ["Hello AURA", "Proxima Centauri", "Warp 7", "CLEARANCE-ALPHA-9", "YES"]
print(f"{'#':<3} {'input':<20} {'before':<14} {'after':<14} slots")
print("-" * 90)
for i, text in enumerate(crew_inputs, 1):
    before = fst.state
    reply = fst.process_utterance(text)
    print(f"{i:<3} {text:<20} {before:<14} {fst.state:<14} {fst.slot_memory}")
    print(f"    AURA: {reply}")

## 2. Noisy channel

Every FST reply is cut to the 7-word budget and carries one noise token. A "word" is a whitespace-delimited token (`str.split()`).

In [ ]:
sample = "I am operating normally under signal degradation right now"
for mode in ("append", "substitute"):
    ch = sim.NoisyChannelSimulator("demo", seed=SEED)
    out = ch.transmit(sample, mode)
    decoded, ok = ch.verify_transmission(out, ch.last_noise_index)
    loss = ch.analyze_semantic_loss(sample, out, ch.last_noise_index)
    print(f"[{mode}]")
    print(f"  raw ({ch.count_words(sample)} words): {sample}")
    print(f"  sent ({ch.count_words(out)} words): {out}")
    print(f"  noise token: {loss['injected_sentinel']} at index {ch.last_noise_index}, valid={ok}")
    print(f"  words kept: {loss['semantic_words_retained']}, cut: {loss['words_truncated']}")

In [ ]:
# Entropy of the noise source over many draws: it approaches log2(7) = 2.8074 bits.
ch = sim.NoisyChannelSimulator("entropy-demo", seed=SEED)
for n in (10, 100, 1000, 10000):
    ch.reset()
    for _ in range(n):
        ch.transmit(sample, NOISE_MODE)
    print(f"n={n:>5}  H(noise) = {ch.channel_entropy():.4f} bits  (max {ch.max_entropy():.4f})")

## 3. Stress tests T1-T5

Each test replays a prefix of the dialogue, then sends one non-canonical input. `hazard_reproduced = True` means the FST under test still shows the documented failure; `False` means it behaves differently (for example, it was fixed).

In [ ]:
results = sim.run_stress_test_suite(FST)
print(f"{'Test':<5} {'Name':<24} {'Input':<30} {'Final state':<14} {'Hazard':<9} Severity")
print("-" * 100)
for r in results:
    print(f"{r['test_id']:<5} {r['name']:<24} {r['trigger_input']:<30} {r['final_state']:<14} "
          f"{str(r['hazard_reproduced']):<9} {r['severity']}")
print()
for r in results:
    print(f"{r['test_id']}: {r['hazard']}  slots={r['recorded_slots']}")

## 4. Live interrogation (8 turns, one conversation)

The eight prompts come from the squad's question bank (Syed Hashir Sohail's four questions plus follow-ups). Each raw reply from the model is passed through the noisy channel; the ledger below shows what the crew would receive. All turns are written to `logs/run_*.jsonl`.

In [ ]:
def resolve_provider(name):
    if name != "auto":
        return name
    if os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY"):
        return "gemini"
    if os.environ.get("OPENAI_API_KEY"):
        return "openai"
    if os.environ.get("ANTHROPIC_API_KEY"):
        return "anthropic"
    return "mock"

provider = resolve_provider(PROVIDER)
llm = sim.LLMClient(provider, use_system_prompt=USE_PERSONA, noise_mode=NOISE_MODE)
channel = sim.NoisyChannelSimulator("Sirius B Transceiver", seed=SEED)
logger = sim.RunLogger(HERE / "logs")
runner = sim.AntiTuringInterrogationRunner(llm, channel, logger)

print("Provider:", provider, "| model:", llm.model, "| noise:", NOISE_MODE, "| history:", HISTORY, "| seed:", SEED)
if provider == "mock":
    print("WARNING: mock provider. Output is canned test text, not model output. Set GEMINI_API_KEY.")

logger.log_event("run_metadata", {"noise_mode": NOISE_MODE, "history": HISTORY, "seed": SEED,
                                  "fst_class": FST.__module__ + "." + FST.__name__, "llm": llm.settings(),
                                  "source": "ps02_solution.ipynb"})
turns = runner.run_all_turns(NOISE_MODE, HISTORY)

print(f"{'Turn':<5} {'ID':<13} {'Raw':<5} {'Kept':<6} Transmitted")
print("-" * 80)
for t in turns:
    print(f"{t['turn']:<5} {t['question_id']:<13} {t['raw_word_count']:<5} "
          f"{t['semantic_retention_rate'] * 100:>3.0f}%   {t['noisy_transmitted_message']}")
print("\nLog file:", logger.log_path)

In [ ]:
print(channel.noise_token_report())

In [ ]:
# Per-turn view: how much of each reply survives, and the running noise entropy.
try:
    import matplotlib.pyplot as plt
    ids = [t["question_id"] for t in turns]
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
    ax[0].bar(ids, [t["semantic_retention_rate"] * 100 for t in turns])
    ax[0].set_ylabel("% of raw reply words kept"); ax[0].set_title("Semantic retention per turn")
    ax[0].tick_params(axis="x", rotation=60)
    ax[1].plot(range(1, len(turns) + 1), [t["cumulative_entropy_bits"] for t in turns], marker="o")
    ax[1].axhline(channel.max_entropy(), linestyle="--")
    ax[1].set_xlabel("turn"); ax[1].set_ylabel("bits"); ax[1].set_title("Cumulative noise entropy (dashed: log2 7)")
    plt.tight_layout(); plt.show()
except ImportError:
    print("matplotlib not installed; skipping plots (pip install matplotlib).")

In [ ]:
# Heuristic flags computed on the RAW replies (not ratings; rate them yourself from the raw text).
for t in turns:
    if t["auto_checks"]:
        print(t["question_id"], t["auto_checks"])

## 5. Save the transcript

In [ ]:
fname = f"05_Noisy_Interrogation_Transcript{SUFFIX}" + ("_MOCK.md" if provider == "mock" else ".md")
# save_transcript never overwrites: if the file exists (for example one you already assessed),
# the new run is written as ..._run2.md, ..._run3.md and so on.
out_path = sim.save_transcript(sim.SCRIPT_DIR / fname,
                               runner.format_markdown_transcript({"noise_mode": NOISE_MODE, "history_mode": HISTORY}))
print("Saved (UTF-8):", out_path)

## 6. Evaluation notes (fill in before sign-off)

- **What the channel does to answers:** the first six words are all the crew receives, so most content is lost. Compare the *Kept* column with the raw replies (for example, whether "408" in turn 8 survives).
- **Ratings per question:** rate Strong / Partial / Weak from the raw replies using the criteria in `01_Sutskever_Inquiry_Brief.md`, and compare with Syed Hashir Sohail's baseline runs.
- **Noise entropy vs meaning:** the entropy values describe the noise source only; they say nothing about whether a reply is correct or creative.
- **Limits:** one conversation, one model, one seed; no tools; ten samples are too few for an entropy estimate near log2 7.